In [27]:
import ee
ee.Authenticate(auth_mode="gcloud")
ee.Initialize(project= 'rmrs-wildfire-treatments')

import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timedelta
import os
import sys
from datetime import datetime
import math
import sklearn
from sklearn.preprocessing import StandardScaler
from google.cloud import storage
from pathlib import Path

In [2]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=tjKJElD0fiXVmrv81kHrYAOnmujo7L&access_type=offline&code_challenge=1OXzCH1iXFp5DX22lqABT6WRtNbxwTp7mYWz2qn_tZw&code_challenge_method=S256


You are now logged in as [hannah.vandusen@usda.gov].
Your current project is [rmrs-wildfire-treatments].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [3]:
!earthengine -h

usage: earthengine [-h] [--ee_config EE_CONFIG]
                   [--service_account_file SERVICE_ACCOUNT_FILE]
                   [--project PROJECT_OVERRIDE]
                   {authenticate,acl,asset,cp,create,ls,alpha,du,mv,model,rm,set_project,task,unset_project,upload,upload_manifest,upload_table_manifest}
                   ...

Earth Engine Command Line Interface.

options:
  -h, --help            show this help message and exit
  --ee_config EE_CONFIG
                        Path to the earthengine configuration file. Defaults
                        to "~/.config\earthengine\credentials".
  --service_account_file SERVICE_ACCOUNT_FILE
                        Path to a service account credentialsfile. Overrides
                        any ee_config if specified.
  --project PROJECT_OVERRIDE
                        Specifies a Google Cloud Platform Project id to
                        override the call.

Commands:
  {authenticate,acl,asset,cp,create,ls,alpha,du,mv,model,rm,set

In [5]:
!earthengine set_project rmrs-wildfire-treatments

Successfully saved project id


In [28]:
# import user-defined settings

# get pathfile of this script
import ipynbname
notebook_path = ipynbname.path()
project_root = notebook_path.parent

import pandas as pd
import ast

# Full local path to user-input csv file
input_path = os.path.join(project_root, "user_input/user_input_general.csv")

# read user-input CSV
df = pd.read_csv(input_path)

# container for created objects from user input csv
user_inputs = {}

for _, row in df.iterrows():
    # Skip rows flagged as R expressions 
    if row["treat_as_R_expression"]: 
        continue
    
    raw = row["value"]

    # Try safe literal parsing; fall back to raw string 
    try: 
        val = ast.literal_eval(raw) 
    except Exception: 
        val = raw 
        
    user_inputs[row["name"]] = val

In [29]:
# Google Bucket name 
bucket_name = 'bb-gee-bucket'

# Bucket folder name
folder_name = 'klamath/PrevFireSev' # to create this folder, must go to https://console.cloud.google.com/storage/browser/bb-gee-bucket/ and add new folder
 
# Local directory to export to
local_directory = user_inputs["processed_data_directory"]
smfires_directory = os.path.join(local_directory, "smfires/raster")


In [15]:
# FUNCTIONS TO CREATE SPECTRAL INDICES

# Function to apply scaling factors
def apply_scale_factors(image):
    optical_bands = image.select(['SR_B.']).multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, None, True)
# Function to apply old scaling factors
def apply_scale_factors2(image):
    optical_bands = image.select(['SR_B.']).multiply(0.001)#.add(-0.2)
    return image.addBands(optical_bands, None, True)

# Function to compute indices for Landsat 8 and 9
def calculate_indices_ls8_9(image):
    nbr = image.normalizedDifference(['SR_B5', 'SR_B7']).toFloat()
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).toFloat()
    ndmi = image.normalizedDifference(['SR_B5', 'SR_B6']).toFloat()

    evi = image.expression(
              '(2.5 * ((B5 - B4) / (B5 + (6 * B4) - (7.5 * B2) + 1)))',
              {'B5': image.select('SR_B5'),
              'B4': image.select('SR_B4'),
              'B2': image.select('SR_B2'),
              }).toFloat()
    mirbi = image.expression(
        '((10 * SR_B6) - (9.8 * SR_B7) + 2)',
        {'SR_B6': image.select('SR_B6'),
         'SR_B7': image.select('SR_B7')}
    ).toFloat()
    qa = image.select(['QA_PIXEL'])
    return nbr.addBands([ndvi, ndmi, evi, mirbi, qa]).select([0, 1, 2, 3, 4, 5], ['nbr', 'ndvi', 'ndmi', 'evi', 'mirbi', 'QA_PIXEL']).copyProperties(image, ['system:time_start'])

# Function to compute indices for Landsat 4, 5, and 7
def calculate_indices_ls4_7(image):
    nbr = image.normalizedDifference(['SR_B4', 'SR_B7']).toFloat()
    ndvi = image.normalizedDifference(['SR_B4', 'SR_B3']).toFloat()
    ndmi = image.normalizedDifference(['SR_B4', 'SR_B5']).toFloat()
    evi = image.expression(
        '2.5 * ((SR_B4 - SR_B3) / (SR_B4 + 6 * SR_B3 - 7.5 * SR_B1 + 1))',
        {'SR_B4': image.select('SR_B4'),
         'SR_B3': image.select('SR_B3'),
         'SR_B1': image.select('SR_B1')}
    ).toFloat()
    mirbi = image.expression(
        '((10 * SR_B5) - (9.8 * SR_B7) + 2)',
        {'SR_B5': image.select('SR_B5'),
         'SR_B7': image.select('SR_B7')}
    ).toFloat()
    qa = image.select(['QA_PIXEL'])
    return nbr.addBands([ndvi, ndmi, evi, mirbi, qa]).select([0, 1, 2, 3, 4, 5], ['nbr', 'ndvi', 'ndmi', 'evi', 'mirbi', 'QA_PIXEL']).copyProperties(image, ['system:time_start'])


# FUNCTION TO MASK CLOUD, WATER, SNOW, ETC.
def lsCfmask(image):
    # Bits 3,4,5,7: cloud,cloud-shadow,snow,water respectively.
    clouds_bit_mask = (1 << 3)
    cloud_shadow_bit_mask = (1 << 4)
    snow_bit_mask = (1 << 5)
    water_bit_mask = (1 << 7)
    # Get the pixel QA band.
    qa = image.select('QA_PIXEL')
    
    # Flags should be set to zero, indicating clear conditions.
    clear = qa.bitwiseAnd(clouds_bit_mask).eq(0).And(qa.bitwiseAnd(cloud_shadow_bit_mask).eq(0)).And(qa.bitwiseAnd(snow_bit_mask).eq(0)).And(qa.bitwiseAnd(water_bit_mask).eq(0))
    return image.updateMask(clear).select([0, 1, 2, 3, 4]).copyProperties(image, ["system:time_start"])


# Main Function for per fire processing
def process_fire(fire): # true/false for using mtbs or local shapefile

    fire_pd = this_fire
    # Date Manipulation of Dataframe
    # Define fire season for AOI
    # Parks et al 2014 used startday = 91 and endday = 181 for fires in Arizona, New Mexico, and Utah.
    # And startday = 152 and endday = 273 for fires in California, Montana, Washington, and Wyoming.
    start_day = user_inputs["start_day_DOY"]
    end_day = user_inputs["end_day_DOY"]
    
    fire_pd['start_day'] = start_day
    fire_pd['end_day'] = end_day

    # All available bands for export
    bandList = ['dnbr', 'rbr', 'rdnbr', 'dndvi', 'devi', 'dndmi', 'dmirbi', 'post_nbr', 'post_mirbi', 'CBI', 'CBI_bc']

    # 2) RANDOM FOREST SPECIFICATIONS------------------------------------------------------------
    # Bands for the random forest classification
    rf_bands = ['def', 'lat', 'rbr', 'dmirbi', 'dndvi', 'post_mirbi']
    
    # Load training data for Random Forest classification
    cbi = ee.FeatureCollection("users/grumpyunclesean/CBI_predictions/data_for_ee_model")
    
    # Load climatic water deficit variable (def) for random forest classification
    def_img = ee.Image("users/grumpyunclesean/CBI_predictions/def").rename('def').int()
    
    # Create latitude image for random forest classification
    lat = ee.Image.pixelLonLat().select('latitude').rename('lat').round().toInt()
    
    # Parameters for random forest classification
    nrow_training_fold = cbi.size()  # number of training observations
    minLeafPopulation = nrow_training_fold.divide(75).divide(6).round()
    
    # Random forest classifier
    fsev_classifier = ee.Classifier.smileRandomForest(
        numberOfTrees = 500,
        minLeafPopulation = minLeafPopulation,
        seed = 123).train(cbi,'CBI',rf_bands).setOutputMode('REGRESSION')

    #  3) GET LANDSAT COLLECTIONS --------------------------------     
    # Landsat 5, 7, 8 and 9 Surface Reflectance (Level 2) Tier 1 Collection 2 
    
    # Get Landsat collections
    ls_collections = {
        'ls9': ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'),
        'ls8': ee.ImageCollection('LANDSAT/LC08/C02/T1_L2'),
        'ls7': ee.ImageCollection('LANDSAT/LE07/C02/T1_L2'),
        'ls5': ee.ImageCollection('LANDSAT/LT05/C02/T1_L2'),
        'ls4': ee.ImageCollection('LANDSAT/LT04/C02/T1_L2')
    }

    # Create water mask from Hansen's Global Forest Change to use in processing function
    water_mask = ee.Image('UMD/hansen/global_forest_change_2023_v1_11').select(['datamask']).eq(1)
    
    # Apply the functions to the collections
    ls_collections = {key: col.map(apply_scale_factors).map(calculate_indices_ls8_9 if key in ['ls8', 'ls9'] else calculate_indices_ls4_7).map(lsCfmask) for key, col in ls_collections.items()}
    
    # Merge Landsat Collections 
    ls_col = ee.ImageCollection(ls_collections['ls9'].merge(ls_collections['ls8']).merge(ls_collections['ls7']).merge(ls_collections['ls5']).merge(ls_collections['ls4']))

    # 4) CALCULATE BURN SEVERITY ON A PER FIRE BASIS -----------------------------------------  
    merged_perim_gdf = fire_pd
    fire_bounds = geemap.geopandas_to_ee(merged_perim_gdf)
        
    fire_year = ee.Date.parse('YYYY', str(fire_pd['Year'].iloc[0]))
    start_day = ee.Number.parse(str(fire_pd['start_day'].iloc[0]))
    end_day = ee.Number.parse(str(fire_pd['end_day'].iloc[0]))


    # Pre-Imagery
    pre_fire_year = fire_year.advance(-1, 'year')
    # Check if imagery is available for 1 year pre fire; if so, get means across pixels; otherwise, output imagery will be masked
    pre_fire_indices = ee.Algorithms.If(
        ls_col.filterBounds(fire_bounds).filterDate(pre_fire_year, fire_year).filter(ee.Filter.dayOfYear(start_day, end_day)).size(),
        ls_col.filterBounds(fire_bounds).filterDate(pre_fire_year, fire_year).filter(ee.Filter.dayOfYear(start_day, end_day)).mean().select([0,1,2,3,4], ['pre_nbr','pre_ndvi', 'pre_ndmi', 'pre_evi','pre_mirbi']),
        ee.Image.cat(ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image()).rename(['pre_nbr','pre_ndvi', 'pre_ndmi', 'pre_evi','pre_mirbi'])
    )

# If any pixels within fire have only one 'scene' or less, add additional year backward to fill in behind
    pre_fire_year2 = fire_year.advance(-2, 'year')
    pre_fire_indices2 = ee.Algorithms.If(
        ls_col.filterBounds(fire_bounds).filterDate(pre_fire_year2, fire_year).filter(ee.Filter.dayOfYear(start_day, end_day)).size(),
        ls_col.filterBounds(fire_bounds).filterDate(pre_fire_year2, fire_year).filter(ee.Filter.dayOfYear(start_day, end_day)).mean().select([0,1,2,3,4], ['pre_nbr','pre_ndvi', 'pre_ndmi', 'pre_evi','pre_mirbi']),
        ee.Image.cat(ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image()).rename(['pre_nbr','pre_ndvi', 'pre_ndmi', 'pre_evi','pre_mirbi'])
    )
    
    pre_filled = ee.Image(pre_fire_indices).unmask(pre_fire_indices2)
    
    # Post-Imagery
    post_fire_year = fire_year.advance(1, 'year')
    post_fire_year2 = fire_year.advance(2, 'year')
    # Check if imagery is available for 1 year post fire; if so, get means across pixels; otherwise, output imagery will be masked
    post_fire_indices = ee.Algorithms.If(
        ls_col.filterBounds(fire_bounds).filterDate(post_fire_year, post_fire_year2).filter(ee.Filter.dayOfYear(start_day, end_day)).size(),
        ls_col.filterBounds(fire_bounds).filterDate(post_fire_year, post_fire_year2).filter(ee.Filter.dayOfYear(start_day, end_day)).mean().select([0,1,2,3,4], ['post_nbr','post_ndvi', 'post_ndmi', 'post_evi','post_mirbi']),
        ee.Image.cat(ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image()).rename(['post_nbr','post_ndvi', 'post_ndmi', 'post_evi','post_mirbi'])
    )
    
    post_fire_year3 = fire_year.advance(3, 'year')
    # If any pixels within fire have only one 'scene' or less, add additional year forward to fill in behind
    post_fire_indices2 = ee.Algorithms.If(
        ls_col.filterBounds(fire_bounds).filterDate(post_fire_year, post_fire_year3).filter(ee.Filter.dayOfYear(start_day, end_day)).size(),
        ls_col.filterBounds(fire_bounds).filterDate(post_fire_year, post_fire_year3).filter(ee.Filter.dayOfYear(start_day, end_day)).mean().select([0,1,2,3,4], ['post_nbr','post_ndvi', 'post_ndmi', 'post_evi','post_mirbi']),
        ee.Image.cat(ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image()).rename(['post_nbr','post_ndvi', 'post_ndmi', 'post_evi','post_mirbi'])
    )
    
    post_filled = ee.Image(post_fire_indices).unmask(post_fire_indices2)
    fireIndices = pre_filled.addBands(post_filled) #Here
    
    # Calculate dNBR
    burnIndices = fireIndices.expression(
        "(b('pre_nbr') - b('post_nbr')) * 1000"
    ).rename('dnbr').toInt().addBands(fireIndices)
    
    # Calculate RBR
    burnIndices2 = burnIndices.expression(
        "b('dnbr') / (b('pre_nbr') + 1.001)"
    ).rename('rbr').toInt().addBands(burnIndices)
    
    # Calculate RdNBR
    burnIndices3 = burnIndices2.expression(
        "abs(b('pre_nbr')) < 0.001 ? 0.001 : b('pre_nbr')"
    ).abs().sqrt().rename('pre_nbr2').toFloat().addBands(burnIndices2)
    
    burnIndices4 = burnIndices3.expression(
        "b('dnbr') / b('pre_nbr2')"
    ).rename('rdnbr').toInt().addBands(burnIndices3)
    
    # Calculate dNDVI
    burnIndices5 = burnIndices4.expression(
        "(b('pre_ndvi') - b('post_ndvi')) * 1000"
    ).rename('dndvi').toInt().addBands(burnIndices4)
    
    # Calculate dEVI
    burnIndices6 = burnIndices5.expression(
        "(b('pre_evi') - b('post_evi')) * 1000"
    ).rename('devi').toInt().addBands(burnIndices5)
    
    # Calculate dNDMI
    burnIndices7 = burnIndices6.expression(
        "(b('pre_ndmi') - b('post_ndmi')) * 1000"
    ).rename('dndmi').toInt().addBands(burnIndices6)
    
    # Calculate dMIRBI
    burnIndices8 = burnIndices7.expression(
        "(b('pre_mirbi') - b('post_mirbi')) * 1000"
    ).rename('dmirbi').toInt().addBands(burnIndices7)
    
    # Multiply post_mirbi band by 1000 to put it on the same scale as CBI plot extractions
    post_mirbi_1000 = burnIndices8.select("post_mirbi").multiply(1000).toInt()
    burnIndices8 = burnIndices8.addBands(post_mirbi_1000, None, True)  # None to copy all bands; True to overwrite original post_mirbi
    
    # Get a list of all metadata properties.
    properties = burnIndices8.propertyNames()
    
    # Add in climatic water deficit variable, i.e. def
    burnIndices9 = burnIndices8.addBands(def_img)
    
    # Add in latitude
    burnIndices10 = burnIndices9.addBands(lat)
    
    # Classify the image with the same bands used to train the Random Forest classifier
    cbi_rf = burnIndices10.select(rf_bands).classify(fsev_classifier).rename('CBI').toFloat().multiply(10**2).floor().divide(10**2)
    
    burnIndices11 = cbi_rf.addBands(burnIndices10)
    
    # Create bias corrected CBI
    def bias_correct(bandName):
        cbi_lo = bandName.expression("((b('CBI') - 1.5) * 1.3)  + 1.5")
        cbi_hi = bandName.expression("((b('CBI') - 1.5) * 1.175) + 1.5")
        cbi_mg = bandName.where(bandName.lte(1.5), cbi_lo).where(bandName.gt(1.5), cbi_hi)
        return cbi_mg.where(cbi_mg.lt(0), 0).where(cbi_mg.gt(3), 3).multiply(10**2).floor().divide(10**2).rename('CBI_bc')
    
    validMask = pre_filled.select('pre_nbr').add(10).add(post_filled.select('post_nbr')).add(10)  # Adding 10 ensures resulting image doesn't have 0 values that would become masked in final output bands
    mask = water_mask.updateMask(validMask)
    burnIndices12 = bias_correct(burnIndices11.select('CBI')).addBands(burnIndices11)
    burnIndices12 = burnIndices12.updateMask(mask).clip(fire_bounds) #.select(bandList, bandList)

    return burnIndices12

In [16]:
import pathlib
shps_pth = os.path.join(local_directory, "smfires/shapefile")
shps_l = list(pathlib.Path(shps_pth).glob('*.shp'))

def get_basenames(file_list):
  return [os.path.basename(file_path).replace('_smfires.shp', '') for file_path in file_list]
ids = get_basenames(shps_l)
ids = list(filter(None, ids))

perims_pth = os.path.join(local_directory, "fire_perimeters/merged")
perims_l = list(pathlib.Path(perims_pth).glob('*.shp'))

perims_l_s = []
for item in perims_l:
    perims_l_s.append(str(item))

def filter_strings(string_list, filter_list):
    return [s for s in string_list if any(substring in s for substring in filter_list)]

perims_l = filter_strings(perims_l_s, ids)

def filter_strings_not(string_list, filter_list):
    return [s for s in string_list if not any(substring in s for substring in filter_list)]

perims_l_not = filter_strings_not(perims_l_s, ids)


In [25]:
# calculate high severity burn threshold

# read in RAVG table
lookup_path = Path(user_inputs["global_data_directory"]) / "ravg_lookup_tables" / "CAModel_Lookup_BA100_EA.txt"

RAVG_basal_area_lookup_table = (
    pd.read_csv(lookup_path, sep=r"\s+")
    .rename(columns=str.lower)
)

# dynamic column name (e.g., "RBR")
sev_col = user_inputs["burn_severity_metric"].lower()

# % basal area loss threshold
perc_BA_loss_threshold = user_inputs["basal_area_high_severity_percent_threshold"]

# get high severity threshold for chosen metric
high_severity_threshold = (
    RAVG_basal_area_lookup_table
    .loc[RAVG_basal_area_lookup_table["meanba"] == perc_BA_loss_threshold, sev_col]
    .min(skipna=True)
)

high_severity_threshold = float(high_severity_threshold)

print(high_severity_threshold)


650.0


In [26]:
for shp, fullp in zip(shps_l, perims_l):
    these_fires = gpd.read_file(shp)
    these_fires['ID'] = range(len(these_fires))
    fire_ids = these_fires['ID'].tolist()

    full_perim = gpd.read_file(fullp)
    
    these_sevs = []

    for fid in fire_ids:
        this_fire = these_fires[these_fires['ID'] == fid].copy()
        this_fire['Year'] = pd.to_datetime(this_fire['FIREYEAR']).dt.year
        this_fire['MTBS_ID'] = this_fire['ID']

        this_sev = process_fire(this_fire)

        these_sevs.append(this_sev)

    collection = ee.ImageCollection.fromImages(these_sevs)
    
    bnds = geemap.geopandas_to_ee(these_fires).geometry().bounds()

    # get maximum severity of prior fires (metric based on user_inputs)
    maxsev = collection.select(user_inputs["burn_severity_metric"].lower()).max().clip(bnds)

    # classify fire severity into low/mod (1) or high (2) based on high severity threshold
    classsev = maxsev.where(maxsev.lte(ee.Number(high_severity_threshold)), 1)
    classsev = classsev.where(classsev.gt(ee.Number(high_severity_threshold)),2)

    classsev = classsev.unmask(0)

    nm = os.path.splitext(os.path.basename(shp))[0]
    ind = nm.find('_')
    nm = nm[:ind]
    nm = nm + '_classified_past_sev'
    # print(nm)

    task = ee.batch.Export.image.toCloudStorage(
        image=classsev,
        description=nm,
        bucket=bucket_name,
        fileNamePrefix=f"{folder_name}/{nm}",
        region=bnds,
        scale=30,
        maxPixels=5e8
    )
    task.start()

    

C:\Users\HannahVanDusen\AppData\Local\Google\Cloud SDK\google-cloud-sdk\platform\bundledpython\pygee\Lib\site-packages\ee\deprecation.py:202: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2023_v1_11! You are using a deprecated asset.
To ensure continued functionality, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2023_v1_11

  warnings.warn(warning, category=DeprecationWarning)


In [27]:
for shp in perims_l_not:
    this_fire = gpd.read_file(shp)
    these_bnds = geemap.geopandas_to_ee(this_fire).geometry().bounds()

    img = ee.Image().clip(these_bnds)

    nm = os.path.splitext(os.path.basename(shp))[0]
    ind = nm.find('_')
    nm = nm[:ind]
    nm = nm + '_classified_past_sev'

    img = img.unmask(0)

    task = ee.batch.Export.image.toCloudStorage(
        image=img,
        description=nm,
        bucket=bucket_name,
        fileNamePrefix=f"{folder_name}/{nm}",
        region=these_bnds,
        scale=30,
        maxPixels=5e8
    )
    task.start()

    

    

In [30]:
storage_client = storage.Client()
bucket = storage_client.bucket(bucket_name)

# Collect expected filenames from earlier
expected_filenames = {
    f"{folder_name}/{os.path.splitext(os.path.basename(shp))[0].split('_')[0]}_classified_past_sev.tif"
    for shp in shps_l + perims_l_not
}

blobs = list(bucket.list_blobs(prefix=folder_name))

for blob in blobs:
    if blob.name in expected_filenames:
        local_path = os.path.join(smfires_directory, os.path.basename(blob.name))
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        print(f"Downloading {blob.name} to {local_path}")
        blob.download_to_filename(local_path)
